In [ ]:
from transformers import AutoTokenizer, Llama4ForConditionalGeneration, FbgemmFp8Config

In [2]:
processor = AutoProcessor.from_pretrained("meta-llama/Llama-4-Scout-17B-16E-Instruct")

`rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1.0 and low_freq_factor=1.0


In [3]:
import torch

In [ ]:
!uv pip install bitsandbytes

/bin/bash: /home/huuthanhvy.nguyen001/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Using Python 3.11.15 environment at: anaconda3/envs/ai
Audited 1 package in 66ms


In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # or torch.bfloat16
    bnb_4bit_use_double_quant=True,        # nested quantization, saves a bit more memory
    bnb_4bit_quant_type="nf4"             # nf4 is better quality than fp4
)

In [6]:
model = AutoModelForImageTextToText.from_pretrained(
    "meta-llama/Llama-4-Scout-17B-16E-Instruct",
    device_map="cuda",
    #dtype=torch.float16,
    quantization_config=bnb_config,
    cache_dir="/hpcstor6/scratch01/h/huuthanhvy.nguyen001"
)

`rope_parameters`'s high_freq_factor field must be greater than low_freq_factor, got high_freq_factor=1.0 and low_freq_factor=1.0


Loading weights:   0%|          | 0/1133 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.50 GiB. GPU 0 has a total capacity of 39.39 GiB of which 724.38 MiB is free. Including non-PyTorch memory, this process has 38.68 GiB memory in use. Of the allocated memory 37.82 GiB is allocated by PyTorch, and 384.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
!nvidia-smi

/bin/bash: /home/huuthanhvy.nguyen001/anaconda3/lib/libtinfo.so.6: no version information available (required by /bin/bash)
Thu Mar 12 10:23:46 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:87:00.0 Off |                    0 |
| N/A   28C    P0              79W / 400W |  29263MiB / 40960MiB |      0%      Default |
|                                 

In [ ]:
import json

with open ("/home/huuthanhvy.nguyen001/newmodel/a1_gen_chatgpt_p0.json", "r") as file:
    data1 = json.load(file)

with open ("/home/huuthanhvy.nguyen001/newmodel/a1_tune_chatgpt_p0.json", "r") as file:
    data2 = json.load(file)

In [ ]:
generate_essay = data1["result"]
tune_essay = data2["result"]

In [ ]:
rubric = {
    "issues": {
        "4": "Issue is stated clearly and described comprehensively, delivering all relevant information necessary for full understanding.",
        "3": "Issue is stated, described, and clarified so that understanding is not seriously impeded by omissions.",
        "2": "Issue is stated but description leaves some terms undefined, ambiguities unexplored, boundaries undetermined, and/or backgrounds unknown.",
        "1": "Issue is stated without clarification or description."
    },
    "evidence": {
        "4": "Information is taken from source(s) with enough interpretation/evaluation to develop a comprehensive analysis or synthesis. Viewpoints of experts are questioned thoroughly.",
        "3": "Information is taken from source(s) with enough interpretation/evaluation to develop a coherent analysis or synthesis. Viewpoints of experts are subject to questioning.",
        "2": "Information is taken from source(s) with some interpretation/evaluation, but not enough to develop a coherent analysis or synthesis. Viewpoints of experts are taken as mostly fact, with little questioning.",
        "1": "Information is taken from source(s) without any interpretation/evaluation. Viewpoints of experts are taken as fact, without question."
    },
    "assumptions": {
        "4": "Thoroughly (systematically and methodically) analyzes own and others' assumptions and carefully evaluates the relevance of contexts when presenting a position.",
        "3": "Identifies own and others' assumptions and several relevant contexts when presenting a position. Questions some assumptions.",
        "2": "Identifies several relevant contexts when presenting a position. May be more aware of others' assumptions than one's own. Shows emerging awareness of present assumptions.",
        "1": "Shows an emerging awareness of present assumptions (sometimes labels assertions as assumptions). Begins to identify some contexts when presenting a position."
    },
    "position": {
        "4": "Specific position is imaginative, taking into account the complexities of an issue. Limits of position are acknowledged. Others' points of view are synthesized within position.",
        "3": "Specific position takes into account the complexities of an issue. Others' points of view are acknowledged within position.",
        "2": "Specific position acknowledges different sides of an issue.",
        "1": "Specific position is stated, but is simplistic and obvious."
    },
    "conclusions": {
        "4": "Conclusions and related outcomes are logical and reflect student's informed evaluation and ability to place evidence and perspectives discussed in priority order.",
        "3": "Conclusion is logically tied to a range of information, including opposing viewpoints; related outcomes are identified clearly.",
        "2": "Conclusion is logically tied to information (chosen to fit the desired conclusion); some related outcomes are identified clearly.",
        "1": "Conclusion is inconsistently tied to some of the information discussed; related outcomes are oversimplified."
    }
}


In [ ]:
messages1 = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": f"""Score the essay on these 5 dimensions, each from 1 to 4:

Rubric:
{json.dumps(rubric, indent=2)}

Scoring scale:
1 = Benchmark
2 = Milestone (lower)
3 = Milestone (upper)
4 = Capstone

Return ONLY a JSON object, no explanation:
{{"issues": <1-4>, "evidence": <1-4>, "assumptions": <1-4>, "position": <1-4>, "conclusions": <1-4>}}

Essay to score:
{generate_essay}"""}
        ]
    },
]

In [ ]:
messages2 = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": f"""Score the essay on these 5 dimensions, each from 1 to 4:

Rubric:
{json.dumps(rubric, indent=2)}

Scoring scale:
1 = Benchmark
2 = Milestone (lower)
3 = Milestone (upper)
4 = Capstone

Return ONLY a JSON object, no explanation:
{{"issues": <1-4>, "evidence": <1-4>, "assumptions": <1-4>, "position": <1-4>, "conclusions": <1-4>}}

Essay to score:
{tune_essay}"""}
        ]
    },
]

In [ ]:
inputs1 = processor.apply_chat_template(
	messages1,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

inputs2 = processor.apply_chat_template(
	messages1,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

In [ ]:
outputs = model.generate(**inputs1, max_new_tokens=40)


KeyboardInterrupt: 

In [ ]:
outputs = model.generate(**inputs2, max_new_tokens=40)


In [ ]:
print(processor.decode(outputs[0][inputs1["input_ids"].shape[-1]:]))  

In [ ]:
print(processor.decode(outputs[0][inputs2["input_ids"].shape[-1]:]))  